# Engine: a chess engine at a chosen strength

**Deliverable:** "a chess engine that plays at a chosen strength" (README, Mark-I scope) and the
analysis that the teaching side (`gambit_teach`) uses to grade the human's moves.

This notebook wraps any UCI engine (Stockfish by default; check its GPL license against the
project's license plan, see README "License") through `python-chess`, plus two tiny pure-Python
engines (`RandomEngine`, `MaterialEngine`) so every other notebook can be tested without a binary.

All engines share one small interface (`EngineLike`):

* `play(board) -> chess.Move` : the move the robot will make.
* `analyse(board, multipv=1, root_moves=None) -> list[Candidate]` : best lines, best first, each
  scored in centipawns **from the point of view of the side to move**.
* `close()`.

**TODO checklist**
- [ ] Install Stockfish (`sudo apt install stockfish` or download) and set `engine.path` in
      `gambit_bringup/config/game.yaml`.
- [ ] Decide how to play below Stockfish's minimum `UCI_Elo` (1320): Skill Level mapping, or the
      teacher policy's own move choice (see `strength_options`).
- [ ] Calibrate the Elo settings against real targets in `run_engine_strength.ipynb` (~1700).
- [ ] Use two engine instances: a strength-limited *player* and a full-strength *analyst*
      (limiting strength also weakens the analysis Stockfish reports).

In [ ]:
from __future__ import annotations

import random
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Protocol, Sequence

import chess
import chess.engine
import yaml

## Configuration

Every setting lives in `gambit_bringup/config/game.yaml`. The path is relative to the notebook's
folder, which works both from `gambit_game/notebooks` and `gambit_teach/notebooks`.

In [ ]:
GAME_CFG_PATH = Path("../../gambit_bringup/config/game.yaml")


def load_game_config(path: Path | str = GAME_CFG_PATH) -> dict:
    """Load game.yaml as a dict."""
    with open(path) as f:
        return yaml.safe_load(f)


GAME_CFG = load_game_config()
ENGINE_CFG = GAME_CFG["engine"]
MATE_SCORE_CP = GAME_CFG["teaching"]["mate_score_cp"]

## Shared types

A `Candidate` is one engine line. `score_cp` is always from the side to move's point of view
(positive = good for the player about to move). A forced mate is turned into a large centipawn
value (`mate_score_cp` minus the distance to mate) so that every score is a plain number.

In [ ]:
@dataclass
class Candidate:
    move: chess.Move
    score_cp: int                       # side-to-move POV
    pv: list = field(default_factory=list)
    mate: Optional[int] = None          # moves to mate (+ we mate, - we get mated), if any

    def __repr__(self) -> str:
        m = f" mate {self.mate}" if self.mate is not None else ""
        return f"Candidate({self.move.uci()}, {self.score_cp:+d}cp{m})"


class EngineLike(Protocol):
    def play(self, board: chess.Board) -> chess.Move: ...
    def analyse(self, board: chess.Board, multipv: int = 1,
                root_moves: Optional[Sequence[chess.Move]] = None) -> list[Candidate]: ...
    def close(self) -> None: ...


def score_to_cp(score: chess.engine.PovScore, pov: chess.Color,
                mate_score: int = MATE_SCORE_CP) -> tuple[int, Optional[int]]:
    """python-chess PovScore -> (centipawns from `pov`, mate distance or None)."""
    s = score.pov(pov)
    return s.score(mate_score=mate_score), s.mate()

## Strength limiting

Stockfish offers two knobs:

* `UCI_LimitStrength=true` + `UCI_Elo=<elo>`: targets a rating (Stockfish accepts roughly 1320..3190;
  the exact range depends on the version, read `engine.options["UCI_Elo"]`).
* `Skill Level` 0..20: older, coarser, not calibrated to Elo.

`strength_options` turns the config into the UCI option dict. It is pure, so it is tested below.

In [ ]:
def strength_options(elo: Optional[int] = None, skill: Optional[int] = None,
                     cfg: dict = ENGINE_CFG) -> dict:
    """UCI options for the requested strength. `skill` wins over `elo`; neither = full strength."""
    if skill is not None:
        if not 0 <= skill <= 20:
            raise ValueError("Skill Level must be in 0..20")
        return {"Skill Level": int(skill)}
    if elo is None:
        return {"UCI_LimitStrength": False}
    if elo < cfg["min_elo"]:
        # TODO: decide how to play below the engine's minimum Elo (Skill Level mapping calibrated in
        # run_engine_strength.ipynb, or let the teacher policy pick weaker multipv moves).
        # For now: clamp to the minimum and let the caller know through the returned value.
        elo = cfg["min_elo"]
    elo = min(int(elo), cfg["max_elo"])
    return {"UCI_LimitStrength": True, "UCI_Elo": elo}

## UCI engine wrapper

`Engine` starts the binary lazily, so constructing it is free and errors appear at the first use.
`find_engine_binary` returns `None` if the binary is not installed (the self-check then skips).

In [ ]:
def find_engine_binary(path: Optional[str] = None) -> Optional[str]:
    """Resolve the configured engine path with shutil.which; None if not installed."""
    path = path or ENGINE_CFG["path"]
    if Path(path).is_file():
        return str(path)
    return shutil.which(path)


class Engine:
    """python-chess UCI engine at a chosen strength.

    Use as a context manager (`with Engine(elo=1700) as eng:`) or call close().
    """

    def __init__(self, path: Optional[str] = None, elo: Optional[int] = None,
                 skill: Optional[int] = None, cfg: dict = ENGINE_CFG):
        self.cfg = cfg
        self.path = path or cfg["path"]
        self.elo = elo
        self.skill = skill if skill is not None else cfg.get("skill_level")
        self._engine: Optional[chess.engine.SimpleEngine] = None

    # -- lifecycle --------------------------------------------------------------
    def open(self) -> "Engine":
        if self._engine is None:
            binary = find_engine_binary(self.path)
            if binary is None:
                raise FileNotFoundError(f"UCI engine '{self.path}' not found (set engine.path in game.yaml)")
            self._engine = chess.engine.SimpleEngine.popen_uci(binary)
            opts = {"Threads": self.cfg["threads"], "Hash": self.cfg["hash_mb"]}
            opts.update(strength_options(self.elo, self.skill, self.cfg))
            self._engine.configure({k: v for k, v in opts.items() if k in self._engine.options})
        return self

    def close(self) -> None:
        if self._engine is not None:
            self._engine.quit()
            self._engine = None

    def __enter__(self):
        return self.open()

    def __exit__(self, *exc):
        self.close()

    def set_strength(self, elo: Optional[int] = None, skill: Optional[int] = None) -> None:
        self.elo, self.skill = elo, skill
        if self._engine is not None:
            opts = strength_options(elo, skill, self.cfg)
            self._engine.configure({k: v for k, v in opts.items() if k in self._engine.options})

    # -- EngineLike ---------------------------------------------------------------
    def play(self, board: chess.Board, time_s: Optional[float] = None) -> chess.Move:
        self.open()
        limit = chess.engine.Limit(time=time_s or self.cfg["think_time_s"])
        return self._engine.play(board, limit).move

    def analyse(self, board: chess.Board, multipv: int = 1,
                root_moves: Optional[Sequence[chess.Move]] = None,
                time_s: Optional[float] = None, depth: Optional[int] = None) -> list[Candidate]:
        self.open()
        depth = depth or self.cfg.get("analysis_depth")
        limit = (chess.engine.Limit(depth=depth) if depth
                 else chess.engine.Limit(time=time_s or self.cfg["analysis_time_s"]))
        infos = self._engine.analyse(board, limit, multipv=multipv,
                                     root_moves=list(root_moves) if root_moves else None)
        if isinstance(infos, dict):
            infos = [infos]
        out = []
        for info in infos:
            if "pv" not in info or not info["pv"]:
                continue
            cp, mate = score_to_cp(info["score"], board.turn)
            out.append(Candidate(info["pv"][0], cp, list(info["pv"]), mate))
        return sorted(out, key=lambda c: -c.score_cp)

## Fallback engines (no binary needed)

* `RandomEngine`: uniformly random legal move; all analysis scores are 0. Useful as a "human".
* `MaterialEngine`: negamax with alpha-beta to a fixed depth on material only (P=100, N=B=300,
  R=500, Q=900). Very weak, but it does take free pieces and see mate-in-one, which is enough to
  exercise the game loop and the teaching code end to end.

In [ ]:
PIECE_CP = {chess.PAWN: 100, chess.KNIGHT: 300, chess.BISHOP: 300,
            chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}


def material_cp(board: chess.Board) -> int:
    """Material balance in centipawns from the side to move's point of view."""
    total = 0
    for piece in board.piece_map().values():
        v = PIECE_CP[piece.piece_type]
        total += v if piece.color == board.turn else -v
    return total


class RandomEngine:
    def __init__(self, seed: Optional[int] = None):
        self.rng = random.Random(seed)

    def play(self, board: chess.Board) -> chess.Move:
        return self.rng.choice(list(board.legal_moves))

    def analyse(self, board, multipv=1, root_moves=None):
        moves = list(root_moves) if root_moves else list(board.legal_moves)
        self.rng.shuffle(moves)
        return [Candidate(m, 0, [m]) for m in moves[:multipv]]

    def close(self):
        pass


class MaterialEngine:
    def __init__(self, depth: int = 2, seed: Optional[int] = 0):
        self.depth = depth
        self.rng = random.Random(seed)

    def _negamax(self, board: chess.Board, depth: int, alpha: int, beta: int, ply: int) -> int:
        if board.is_checkmate():
            return -(MATE_SCORE_CP - ply)
        if board.is_stalemate() or board.is_insufficient_material():
            return 0
        if depth == 0:
            return material_cp(board)
        best = -10 * MATE_SCORE_CP
        # captures first: better pruning
        for move in sorted(board.legal_moves, key=lambda m: not board.is_capture(m)):
            board.push(move)
            val = -self._negamax(board, depth - 1, -beta, -alpha, ply + 1)
            board.pop()
            best = max(best, val)
            alpha = max(alpha, val)
            if alpha >= beta:
                break
        return best

    def analyse(self, board, multipv=1, root_moves=None):
        b = board.copy(stack=False)
        moves = list(root_moves) if root_moves else list(b.legal_moves)
        self.rng.shuffle(moves)                     # random tie-break between equal moves
        scored = []
        for m in moves:
            b.push(m)
            # full window per root move so every multipv score is exact
            val = -self._negamax(b, self.depth - 1, -10 * MATE_SCORE_CP, 10 * MATE_SCORE_CP, 1)
            b.pop()
            mate = None
            if abs(val) >= MATE_SCORE_CP - 100:
                plies = MATE_SCORE_CP - abs(val)
                mate = (plies + 1) // 2 if val > 0 else -((plies + 1) // 2)
            scored.append(Candidate(m, val, [m], mate))
        scored.sort(key=lambda c: -c.score_cp)
        return scored[:multipv]

    def play(self, board):
        return self.analyse(board, multipv=1)[0].move

    def close(self):
        pass


def make_engine(prefer_uci: bool = True, elo: Optional[int] = None) -> EngineLike:
    """Stockfish if installed (and preferred), else MaterialEngine."""
    if prefer_uci and find_engine_binary() is not None:
        return Engine(elo=elo)
    return MaterialEngine()

## Self-check

In [ ]:
assert strength_options(1700) == {"UCI_LimitStrength": True, "UCI_Elo": 1700}
assert strength_options(800)["UCI_Elo"] == ENGINE_CFG["min_elo"]      # clamped (see TODO)
assert strength_options(skill=5) == {"Skill Level": 5}
assert strength_options() == {"UCI_LimitStrength": False}

# MaterialEngine takes a hanging queen: 1.e4 d5 2.Qh5?? ... the black knight can't, set up directly
b = chess.Board("4k3/8/8/3q4/8/8/8/3QK3 w - - 0 1")      # white queen takes black queen on d5
assert MaterialEngine().play(b) == chess.Move.from_uci("d1d5")
# mate in one: back-rank
b = chess.Board("6k1/5ppp/8/8/8/8/8/R5K1 w - - 0 1")
best = MaterialEngine().analyse(b, multipv=3)
assert best[0].move == chess.Move.from_uci("a1a8") and best[0].mate == 1, best
assert len(best) == 3 and best[0].score_cp >= best[1].score_cp
# root_moves restricts the search
only = MaterialEngine().analyse(b, root_moves=[chess.Move.from_uci("g1f1")])
assert len(only) == 1 and only[0].move.uci() == "g1f1"
assert RandomEngine(0).play(chess.Board()) in chess.Board().legal_moves

if find_engine_binary():
    with Engine(elo=1700) as eng:
        cands = eng.analyse(chess.Board(), multipv=3, time_s=0.05)
        assert len(cands) == 3 and cands[0].move in chess.Board().legal_moves
    print("Stockfish check passed")
else:
    print("Stockfish not installed: UCI check skipped")
print("engine self-check passed")